# NLP Sentence Similarity — Tech Persona (rebuilt pipeline)

**What changed from the original version** (see README.md for the full list):

- `bert_serving` / `BertClient` is gone. Embeddings come from a **pinned
  sentence-transformers checkpoint** (`sentence-transformers/all-MiniLM-L6-v2`,
  revision pinned in `sentence_similarity.py`), using its configured
  **mean pooling**, with L2-normalised embeddings.
- Messages are embedded **whole**, not word-by-word, so context — including
  negation — is kept.
- Scoring/evaluation code lives in `sentence_similarity.py`; this notebook
  builds a **labelled dataset**, splits it train/validation/test, chooses the
  decision threshold on **validation** data only, and reports **accuracy,
  AUC and F1 on the test set** for the transformer score and two baselines
  (TF-IDF cosine, averaged word vectors).
- **Lift** is computed per token **position**, so repeated words are handled
  individually.
- No measured numbers are hard-coded in the text: every figure below is
  computed by the cells when the notebook is run.

# Introduction

The purpose of this project is to create a persona with a technical
personality. The persona should be able to differentiate a technical message
from a message on any other topic, on the basis of an interest score. To
achieve this, I first create a sample dictionary of tech terms through which
our input message is compared. **The dictionary has 59 entries, of which 58
are unique** (the original list contained "electronics" twice; the earlier
text saying "approximately 50" words was wrong). The duplicate has been
removed, "headhopnes" has been corrected to "headphones", and capitalisation
has been normalised ("TV" → "tv", "Camera" → "camera").

Word embedding is a technique in NLP where words or phrases are mapped to
vectors of real numbers. BERT is a technique for NLP pre-training developed
by Google to understand user searches. BERT uses a Transformer, an attention
mechanism that learns contextual relations between words (or sub-words) in a
text.

Once we have vectors, we calculate similarity using cosine similarity — a
measure of similarity between two non-zero vectors that measures the cosine
of the angle between them.

**Design change, and why:** the original version embedded each filtered word
separately and took its maximum cosine against every dictionary word. That
design — not BERT — caused the weak separation reported originally: a message
word identical to a dictionary entry scores cosine 1.0 by construction, and
averaging per-word maxima gives even unrelated messages high scores (all of
the original examples, tech and non-tech alike, scored between about 78% and
90%). This version embeds the whole message once and compares it with a
single dictionary prototype (the mean of the dictionary embeddings), and it
evaluates that score properly on held-out data (below).

The pipeline has 4 steps:

1. Preprocessing (negations kept — see below)
2. Calculating interest scores (whole message vs. dictionary prototype)
3. Calculating the **lift** scores (leave-one-out, by token position)
4. Visualising positive and negative lift scores

In [ ]:
# Setup. The model is downloaded once from Hugging Face and cached.
# Checkpoint + pooling are documented in sentence_similarity.py and printed here.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

import nltk

import sentence_similarity as ss

# NLTK data used by preprocessing (one-time download).
for _pkg in ("punkt", "punkt_tab", "stopwords"):
    nltk.download(_pkg, quiet=True)

print("Model:", ss.MODEL_NAME)
print("Pinned revision:", ss.MODEL_REVISION)
print("Pooling:", ss.POOLING)

encoder = ss.load_encoder()
prototype = ss.dictionary_prototype(encoder)  # dictionary embedded in ONE batched call

In [ ]:
# The tech dictionary: 59 entries originally, 58 unique after removing the
# duplicate "electronics"; spelling and case fixed in sentence_similarity.py.
print(len(ss.DICTIONARY), "unique dictionary entries")
ss.DICTIONARY

# Steps and Logic

1) **Preprocessing:** convert to lower case, tokenise, remove punctuation and
stop words. One correction from the original version: **negations are kept.**
NLTK's English stopword list includes "not" and "no", so the original code
deleted them and flipped the meaning of messages like "I do not like gaming".
(The transformer score is computed from the raw message, since the encoder
uses stopwords and word order for context; this tokenisation feeds the
baselines and the lift analysis.)

2) **Interest scores:** the *whole* preprocessed-context message is embedded
in a single (batched) encode call and compared by cosine similarity with the
dictionary prototype. There is no per-word / per-dictionary-pair encode loop.

3) **Lift scores:** to calculate lift, one token at a time is deleted and the
interest score is recomputed; lift = original score − score without that
token. Deletion is **by position**, so if a word occurs three times, each
occurrence gets its own lift value (the original code grouped by the word
string and collapsed repeats).

4) **Visualisations:** if there are at least 3 positive and 3 negative lift
values, bar plots of both are returned.

In [ ]:
# Preprocessing demo — note that "not" and "no" survive.
print(ss.preprocess("I do NOT like gaming, and no update will change that!"))

In [ ]:
def persona(message):
    """
    Score one message and show its per-position lift.

    :param message: input message text
    :returns: (interest score, DataFrame of lift by token position)
    """
    score = ss.persona(message, encoder, prototype)
    lift = ss.lift_by_position(message, encoder, prototype)
    print(f"Interest score: {score:.3f}")
    pos = lift[lift["lift"] > 0].sort_values("lift")
    neg = lift[lift["lift"] < 0].sort_values("lift")
    if len(pos) >= 3 and len(neg) >= 3:
        pos.plot.barh(x="word", y="lift", legend=False, title="Positive lift by word position")
        plt.show()
        neg.plot.barh(x="word", y="lift", legend=False, title="Negative lift by word position")
        plt.show()
    return score, lift

In [ ]:
# Lift by position: "phone" occurs at three positions and each occurrence
# is removed and scored separately (the old groupby('word') code could not do this).
score, lift = persona("I love my phone, my phone camera is great and my phone battery lasts all day")
lift

# Evaluation — a labelled dataset, kept separate from the dictionary

The original notebook "tested" the function on a handful of example messages
with no labels and no held-out data — and several of those examples were
built out of dictionary words, which under the old per-word max-cosine design
guaranteed inflated scores. Below is a small hand-labelled coursework
dataset (24 tech / 24 non-tech messages). Rules enforced by the next cell:

- **No message is a dictionary entry**, and the dictionary is used only to
  build the prototype — it is never part of the train/validation/test data.
- The split is stratified, with a fixed seed; the threshold is chosen on the
  **validation** split only, and the **test** split is scored once.

In [ ]:
LABELED = [
    # --- tech (1) ---
    ("My laptop battery drains fast since the latest software update", 1),
    ("I spent the weekend building a gaming pc with a new graphics card", 1),
    ("The new iphone camera takes amazing photos in low light", 1),
    ("Our team migrated the database to the cloud last week", 1),
    ("I need a mechanical keyboard for long coding sessions", 1),
    ("The bluetooth headphones keep disconnecting from my phone", 1),
    ("He is studying machine learning and data science at university", 1),
    ("The wifi router needs a firmware update to fix the speed", 1),
    ("I bought a tablet mainly for reading and watching videos", 1),
    ("Version control with git makes collaborating on code much easier", 1),
    ("The android app crashes whenever I open the settings page", 1),
    ("A solid state drive made my old computer feel new again", 1),
    ("We set up a home theater with surround speakers in the basement", 1),
    ("The sensor in the smartwatch tracks my heart rate all day", 1),
    ("She develops mobile apps for a software company downtown", 1),
    ("Online gaming with friends is how I spend most weekends", 1),
    ("The printer at work keeps jamming on the top floor", 1),
    ("Artificial intelligence tools now help doctors read scans", 1),
    ("I charge my phone overnight so it is full in the morning", 1),
    ("Streaming the match on my television beats watching on a small screen", 1),
    ("Backing up photos to cloud storage saved me when my drive failed", 1),
    ("The gaming tournament finals went to a deciding fifth map", 1),
    ("A wireless mouse and a second monitor improved my desk setup", 1),
    ("Learning to code in python opened up a career in technology", 1),
    # --- non-tech (0) ---
    ("This world is a better place without terrorism and violence", 0),
    ("Health is wealth, so eat green vegetables and exercise daily", 0),
    ("This world is a better place without diseases that lead to death", 0),
    ("The garden needs watering before the tomatoes will grow", 0),
    ("She baked a chocolate cake for her daughter's birthday party", 0),
    ("The train to the city was delayed by heavy snow this morning", 0),
    ("We are planning a beach holiday with the whole family in June", 0),
    ("The novel's final chapter tied up every loose end beautifully", 0),
    ("Regular exercise and enough sleep keep my energy up", 0),
    ("The committee debated the new housing policy for three hours", 0),
    ("Fresh basil and tomatoes make a simple pasta sauce", 0),
    ("The museum's new exhibition covers two centuries of painting", 0),
    ("Interest rates rose again, worrying first-time home buyers", 0),
    ("The puppy learned to sit and stay within a week", 0),
    ("Rain flooded several streets in the old part of town", 0),
    ("He practices the piano for an hour before school every day", 0),
    ("The farmers market sells honey, cheese and fresh bread", 0),
    ("Meditation in the morning helps me stay calm under pressure", 0),
    ("The football coach praised the team's defending after the match", 0),
    ("Volunteers served soup at the shelter throughout the winter", 0),
    ("The recipe calls for two cups of flour and a pinch of salt", 0),
    ("Grandma tells wonderful stories about growing up in the village", 0),
    ("The orchestra performed Beethoven to a full concert hall", 0),
    ("I have no interest in computers or phones at all", 0),
]

messages = [m for m, _ in LABELED]
labels = np.array([y for _, y in LABELED])

# Separation checks: dictionary != evaluation data.
dict_set = set(ss.DICTIONARY)
assert not any(m.lower() in dict_set for m in messages), "a test message is a dictionary entry"
print(f"{len(messages)} labelled messages: {labels.sum()} tech / {(labels == 0).sum()} non-tech")

In [ ]:
# Stratified train / validation / test split, fixed seed.
# Threshold is tuned on validation ONLY; test is touched once, at the end.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    messages, labels, test_size=0.40, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)
print(f"train {len(X_train)} / validation {len(X_val)} / test {len(X_test)}")

## Three scorers

1. **Whole-message transformer score** — cosine between the whole-message
   embedding and the dictionary prototype (the rebuilt method).
2. **TF-IDF cosine baseline** — vectoriser fitted on the dictionary entries;
   a message scores its maximum cosine against a dictionary entry.
3. **Averaged word-vector baseline** — preprocessed words embedded in one
   batched call, averaged, compared with the same prototype. This discards
   word order and context, which is exactly the contrast of interest.

Every scorer runs on all three splits; embeddings for each scorer are
computed in batched calls, not per-pair loops.

In [ ]:
def all_scores(texts):
    return {
        "transformer (whole message)": ss.interest_scores(texts, encoder, prototype),
        "tfidf cosine": ss.tfidf_scores(texts),
        "averaged word vectors": ss.averaged_word_vector_scores(texts, encoder, prototype),
    }

scores = {name: all_scores(texts) for name, texts in
          [("train", X_train), ("val", X_val), ("test", X_test)]}
scores["train"].keys()

In [ ]:
# Choose each method's threshold on the VALIDATION split (best F1).
thresholds = {}
for method in scores["val"]:
    cand = np.unique(np.round(scores["val"][method], 3))
    best_t, best_f1 = 0.5, -1.0
    for t in cand:
        f1 = f1_score(y_val, (scores["val"][method] >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), f1
    thresholds[method] = best_t
    print(f"{method}: validation threshold = {best_t:.3f} (validation F1 = {best_f1:.3f})")

In [ ]:
# Score the TEST split once, with the validation-chosen thresholds.
rows = []
for method, test_scores in scores["test"].items():
    pred = (test_scores >= thresholds[method]).astype(int)
    rows.append({
        "method": method,
        "accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, test_scores),
        "F1": f1_score(y_test, pred, zero_division=0),
    })
# Majority-class reference point for the test split.
majority_acc = max(np.mean(y_test), 1 - np.mean(y_test))
print(f"Majority-class baseline accuracy on test: {majority_acc:.3f}")
pd.DataFrame(rows)

# Conclusion (corrected)

The original conclusion claimed interest scores of "85% or higher" for
tech-related messages and "75% to 82%" for unrelated ones. **Its own examples
contradict that:** a non-tech message about terrorism was reported at about
82% (82.35% in the run output) — at/above the top of the claimed non-tech
range and close to the tech examples (87–90%) — and the other non-tech
examples scored 78.4% and 78.8%, inside the same narrow band as everything
else. With scores that close together, no threshold separates the two groups
well, which is what the evaluation above now measures directly instead of
asserting ranges.

The original text also said "BERT is not very good in predicting the
semantics of different sentences". That blamed the model for a **design
choice in this project**: the original code embedded single words, took each
word's *maximum* cosine against the dictionary (so exact dictionary matches
scored 1.0), and averaged — a method that discards sentence context by
construction. Modern sentence-transformer checkpoints are trained precisely
for sentence-level semantics; whether this persona score separates tech from
non-tech messages is an empirical question, answered by the held-out
accuracy / AUC / F1 table above (computed when the notebook runs) rather
than by the anecdotal examples.

Limitations: the labelled dataset is small and hand-written for coursework,
so the metrics have wide uncertainty and the validation-chosen threshold is
only as good as those 48 messages. A larger, independently labelled dataset
is the obvious next step.

# Next Steps

- Grow the labelled dataset (ideally messages written/collected independently
  of the dictionary) and re-tune the threshold.
- Try other sentence encoders (larger MiniLM / MPNet checkpoints) and compare
  on the same split, using the same threshold-on-validation protocol.
- Compare against the classical baselines already included here (TF-IDF,
  averaged word vectors) before assuming the transformer is needed.
- Classic static embeddings (word2vec, GloVe, fastText) can be added as
  further averaged-vector baselines under the same evaluation.